# Model Performance Analysis Across All Rounds and Multiple Random Seeds

# 1 Shared Parameters and Data Preparation

## 1.1 Shared Parameters and Data Loading

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV
from xgboost import XGBRegressor
from IPython.display import clear_output


# Set production inputs, outputs, and a consistent feature order; replace only this parameter block for example runs.
PROJECT_ROOT = Path.cwd()

DATA_PATH = PROJECT_ROOT / "Example" / "ALL_data_analysis" / "ALL_exp._results.csv" 
OUTPUT_PATH = PROJECT_ROOT / "Example" / "ALL_data_analysis" / "model_analysis"

ANALYSIS_ROUND = ["Round 1","Round 2","Round 3","Round 4"] 

all_data = pd.read_csv(DATA_PATH)
input_files = {}
for round in ANALYSIS_ROUND:    
    input_files[round] = all_data[all_data["Round"]==round]

# YE (g/L)	Tryptone (g/L)	NaCl (g/L)	K2HPO4(g/L)	MgSO4·7H2O (g/L)	Glycerol (g/L)	FAC (g/L)	Na2S2O3 (g/L)	Triton X-100 (g/L)	Glycine (g/L)	NH4OAc (g/L)	CSL-P (g/L)	ZnSO4·7H2O (g/L)	Methionine (g/L)	Cysteine (g/L)	(NH4)2SO4 (g/L)
# Standardize the target and features. EGT titer (mg/L)	 Medium material cost (CNY/L)	EGT efficiency cost (mg /CNY)
TARGET_COLUMN = "EGT titer (mg/L)"
FEATURE_COLUMNS = ['YE (g/L)','Tryptone (g/L)','NaCl (g/L)',
                    'K2HPO4 (g/L)','MgSO4·7H2O (g/L)','Glycerol (g/L)','FAC (g/L)',
                    'Na2S2O3 (g/L)','Triton X-100 (g/L)','Glycine (g/L)',
                    'NH4OAc (g/L)','CSL-P (g/L)','ZnSO4·7H2O (g/L)','Methionine (g/L)',
                    'Cysteine (g/L)','(NH4)2SO4 (g/L)']  
# Specify any number of random seeds as a list, along with model selection criteria and parallel computing parameters.
random_seeds = [0, 1, 42, 123, 256, 512, 1024, 2023, 2024, 2025]
# test ：  [0, 1]  

ensemble_size = 20
search_iterations = 200
cv_fold_count = 5

parameter_distributions = {
    "learning_rate": [0.01, 0.03, 0.1, 0.3],
    "colsample_bytree": [0.6, 0.8, 0.9, 1.0],
    "subsample": [0.6, 0.8, 0.9, 1.0],
    "max_depth": [2, 3, 4, 6, 8],
    "n_estimators": [10, 20, 40, 60, 80, 100, 300, 500],
    "reg_lambda": [1, 1.5, 2],
    "gamma": [0, 0.1, 0.4, 0.6],
    "min_child_weight": [1, 2, 4],
}

# Create the output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Load data for each round, validate worksheets and required columns, and standardize the 17 volume features, target values, and source row numbers.
round_data = {}
source_records = []
for round_name, source_frame in input_files.items():

    required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]

    round_frame = source_frame[required_columns].copy()

    round_frame.insert(0, "source_row", np.arange(2, len(round_frame) + 2))
    round_frame.insert(0, "round", round_name)
    round_data[round_name] = round_frame
    source_records.append({
        "round": round_name,
        "input_path": str(DATA_PATH),
        "modified_time": pd.Timestamp(DATA_PATH.stat().st_mtime, unit="s"),
        "row_count": len(source_frame),
        "column_count": source_frame.shape[1],
        "field_names": ", ".join(source_frame.columns.astype(str)),
    })

# Display key parameters affecting the results and input provenance.
run_parameters = pd.DataFrame({
    "parameter": [
        "random_seeds", "seed_count", "search_iterations", "cv_fold_count", "ensemble_size", "evaluation_round",
    ],
    "value": [
        str(random_seeds), len(random_seeds), search_iterations, cv_fold_count, ensemble_size, "round4",
    ],
})
source_info = pd.DataFrame(source_records)
display(run_parameters)
display(source_info)

,parameter,value
0,random_seeds,"[0, 1, 42, 123, 256, 512, 1024, 2023, 2024, 2025]"
1,seed_count,10
2,search_iterations,200
3,cv_fold_count,5
4,ensemble_size,20
5,evaluation_round,round4


,round,input_path,modified_time,row_count,column_count,field_names
0,Round 1,c:\Users\Liaoyt\Desktop\Ai_medium\github\Examp...,2026-09-18 02:09:34.851810455,90,59,"Round, YE (g/L), Tryptone (g/L), NaCl (g/L), K..."
1,Round 2,c:\Users\Liaoyt\Desktop\Ai_medium\github\Examp...,2026-09-18 02:09:34.851810455,90,59,"Round, YE (g/L), Tryptone (g/L), NaCl (g/L), K..."
2,Round 3,c:\Users\Liaoyt\Desktop\Ai_medium\github\Examp...,2026-09-18 02:09:34.851810455,90,59,"Round, YE (g/L), Tryptone (g/L), NaCl (g/L), K..."
3,Round 4,c:\Users\Liaoyt\Desktop\Ai_medium\github\Examp...,2026-09-18 02:09:34.851810455,90,59,"Round, YE (g/L), Tryptone (g/L), NaCl (g/L), K..."


# 2 Three-Stage Training and Round 4 Evaluation Across Multiple Random Seeds

Iterate over each seed in the shared list `random_seeds` and complete three cumulative training stages. At each stage, randomly sample 200 candidate parameter sets and perform 5-fold cross-validation on all accumulated historical data, selecting the top 20 sets by mean MAE. Record the mean R², Spearman correlation, and MAE of the selected sets, then train 20 members on all accumulated data and predict Round4 only. Both per-seed results and cross-seed summaries use dynamic long-format tables; all expected row counts are calculated from `len(random_seeds)`.

## 2.1 Data Processing

In [7]:
def fit_ensemble(X, y, parameter_list, random_seed):
    """Train a 20-member XGBoost ensemble using the given parameter sets and random seed."""
    return [
        XGBRegressor(
            objective="reg:squarederror",
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ).fit(X, y)
        for parameters in parameter_list
    ]


def ensemble_predict(models, X):
    """Compute the per-sample mean of ensemble member predictions."""
    return np.mean([model.predict(X) for model in models], axis=0)


def calculate_spearman(actual, predicted):
    """Compute the Spearman rank correlation between observed and predicted values."""
    return float(spearmanr(actual, predicted).statistic)


# Define three cumulative training stages and model selection metrics.
stage_specs = [
    {"model": "Round1", "training_rounds": ["Round 1"]},
    {"model": "Round1+2", "training_rounds": ["Round 1", "Round 2"]},
    {"model": "Round1+2+3", "training_rounds": ["Round 1", "Round 2", "Round 3"]},
]
cv_scoring = {
    "r2": "r2",
    "spearman": make_scorer(calculate_spearman),
    "mae": "neg_mean_absolute_error",
}
metric_records = []

# Use round 4 as the test set throughout
evaluation_data = round_data["Round 4"]
evaluation_X = evaluation_data[FEATURE_COLUMNS]
evaluation_y = evaluation_data[TARGET_COLUMN].to_numpy()

# Evaluate each model using its own recommended samples as the test set
next_data = ["Round 2","Round 3","Round 4"]
next_id = 0


# Perform parameter selection, ensemble retraining, and Round 4 evaluation for each seed and each of the three cumulative training stages.
for stage in stage_specs:

    next_id += 1 
    
    for random_seed in random_seeds:
        clear_output()
        print("model:{}".format(stage["model"]))
        print(f"random_seed:{random_seed}")
        # Retrieve the training set from the stage dictionary
        training_data = pd.concat(
            [round_data[name] for name in stage["training_rounds"]],
            ignore_index=True,
        )
        X_full = training_data[FEATURE_COLUMNS]
        y_full = training_data[TARGET_COLUMN]

        # Use the same seed for parameter sampling, cross-validation splits, and XGBoost randomness.
        search = RandomizedSearchCV(
            estimator=XGBRegressor(
                objective="reg:squarederror",
                random_state=random_seed,
                n_jobs=-1,
            ),
            param_distributions=parameter_distributions,
            n_iter=search_iterations,
            scoring=cv_scoring,
            cv=KFold(
                n_splits=cv_fold_count,
                shuffle=True,
                random_state=random_seed,
            ),
            random_state=random_seed,
            n_jobs=-1,
            refit=False,
            error_score="raise",
        )
        search.fit(X_full, y_full)
        search_results = pd.DataFrame(search.cv_results_)

        # Select the top 20 parameter sets by mean 5-fold MAE and average the three metrics over the selected sets.
        selected_results = (
            search_results
            .sort_values("mean_test_mae",ascending=False,kind="stable",)
            .head(ensemble_size)
            .copy()
        )

        top_parameters = selected_results["params"].tolist()
        
        selected_top20_cv_r2_mean = selected_results["mean_test_r2"].mean()
        selected_top20_cv_spearman_mean = selected_results["mean_test_spearman"].mean()
        selected_top20_cv_mae_mean = -selected_results["mean_test_mae"].mean()

        # Retrain 20 members on the accumulated training data and compute ensemble predictions and evaluation metrics for Round 4 only.
        final_models = fit_ensemble(X_full, y_full, top_parameters, random_seed)
        round4_predicted_yield = ensemble_predict(final_models, evaluation_X)

        evaluation_data = round_data[next_data[next_id-1]]
        next_X = evaluation_data[FEATURE_COLUMNS]
        next_y = evaluation_data[TARGET_COLUMN].to_numpy()

        next_predicted_yield = ensemble_predict(final_models,next_X)

        # Summarize final results
        metric_records.append({
            "random_seed": random_seed,
            "model": stage["model"],
            "training_rounds": "+".join(stage["training_rounds"]),

            "selected_top20_cv_r2_mean": selected_top20_cv_r2_mean,
            "selected_top20_cv_spearman_mean": selected_top20_cv_spearman_mean,
            "selected_top20_cv_mae_mean": selected_top20_cv_mae_mean,

            "round4_r2": r2_score(evaluation_y, round4_predicted_yield),
            "round4_spearman": calculate_spearman(evaluation_y, round4_predicted_yield),
            "round4_mae": mean_absolute_error(evaluation_y, round4_predicted_yield),

            "next_r2": r2_score(next_y, next_predicted_yield),
            "next_spearman": calculate_spearman(next_y, next_predicted_yield),
            "next_mae": mean_absolute_error(next_y, next_predicted_yield),

            "training_n": len(training_data),
            "round4_n": len(evaluation_data),
            "cv_fold_count": cv_fold_count,
            "searched_parameter_count": len(search_results),
            "selected_parameter_count": len(selected_results),
            "selection_metric": "cv_mae_mean_ascending",
        })

# Create an internal wide-format table with one row per seed and model, and validate the result granularity.
seed_model_metrics_wide = pd.DataFrame(metric_records)
seed_model_metrics_wide.to_csv(OUTPUT_PATH/"seed_model_metrics_wide.csv")





model:Round1+2+3
random_seed:2025


## 2.1 Mean and Significance Analysis

In [10]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

# ===============Calculate Score Means ± Standard Errors=====================
score_list = ["selected_top20_cv_r2_mean",
            "selected_top20_cv_spearman_mean",
            "selected_top20_cv_mae_mean",
            "round4_r2",
            "round4_spearman",
            "round4_mae",
            "next_r2",
            "next_spearman",
            "next_mae",
            ]

record = []
for stage in stage_specs:
    
    score_record = []
    score_record.append(stage["model"])
    for score in score_list:
        scores = seed_model_metrics_wide[seed_model_metrics_wide["model"]==stage["model"]][score]

        n = len(scores)
        mean = np.mean(scores)                  # Mean
        std_sample = np.std(scores, ddof=1)     # Sample standard deviation
        sem = std_sample / np.sqrt(n)           # Standard error (unused)

        score_record.append(f"{mean:.2f} ± {std_sample:.2f}")

    record.append(score_record)
record = np.array(record)

selected_top20_cv = pd.DataFrame(record[:,0:4],columns=["model", "cv_r2", "cv_spearman","cv_mae"])
round4 = pd.DataFrame(record[:,[0,*range(4,7)]],columns=["model", "round4_r2", "round4_spearman","round4_mae"])
next = pd.DataFrame(record[:,[0,*range(7,10)]],columns=["model", "next_r2", "next_spearman","next_mae"])

display(selected_top20_cv)
display(round4)
display(next)


,model,cv_r2,cv_spearman,cv_mae
0,Round1,0.74 ± 0.04,0.82 ± 0.02,38.46 ± 1.61
1,Round1+2,0.82 ± 0.02,0.87 ± 0.01,29.55 ± 0.85
2,Round1+2+3,0.84 ± 0.01,0.82 ± 0.01,26.74 ± 0.68


,model,round4_r2,round4_spearman,round4_mae
0,Round1,-0.15 ± 0.09,0.33 ± 0.04,39.52 ± 1.59
1,Round1+2,0.32 ± 0.03,0.55 ± 0.01,28.76 ± 0.51
2,Round1+2+3,0.37 ± 0.02,0.61 ± 0.00,27.36 ± 0.32


,model,next_r2,next_spearman,next_mae
0,Round1,0.19 ± 0.04,0.50 ± 0.03,32.51 ± 0.99
1,Round1+2,-0.61 ± 0.07,0.28 ± 0.03,27.02 ± 0.74
2,Round1+2+3,0.37 ± 0.02,0.61 ± 0.00,27.36 ± 0.32
